# 3. Data Validation (Validación de Datos)

## Objetivo
Verificar que los datos limpios cumplan con todos los criterios de calidad:
- Integridad de estructura
- Validez de valores
- Consistencia referencial
- Documentar cambios realizados

## Entrada
Datos limpios de `data/processed/traffic_data_clean.csv`

## Salida
Dataset validado y reporte de calidad

## 3.1 Importar librerías

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

print('✅ Librerías importadas')

## 3.2 Cargar datos limpios

In [ ]:
# Cargar datos limpios
CLEAN_PATH = '../data/processed/traffic_data_clean.csv'
df_clean = pd.read_csv(CLEAN_PATH, parse_dates=['timestamp'])

print(f'✅ Datos limpios cargados: {len(df_clean)} registros')
print(f'Dimensiones: {df_clean.shape}')

## 3.3 Validación 1: Estructura

In [ ]:
print('VALIDACIÓN 1: ESTRUCTURA DE DATOS\n')

# Columnas esperadas
columnas_esperadas = ['timestamp', 'avenida', 'latitud', 'longitud', 
                      'velocidad', 'densidad', 'detenciones']

columnas_faltantes = [col for col in columnas_esperadas if col not in df_clean.columns]

if columnas_faltantes:
    print(f'🔴 ADVERTENCIA: Columnas faltantes: {columnas_faltantes}')
else:
    print('✅ Todas las columnas esperadas están presentes')

print(f'\nColumnas presentes ({len(df_clean.columns)}):')  
for col in df_clean.columns:
    print(f'  - {col}: {df_clean[col].dtype}')

## 3.4 Validación 2: Completitud de datos

In [ ]:
print('VALIDACIÓN 2: COMPLETITUD DE DATOS\n')

# Verificar valores nulos
nulos_totales = df_clean.isnull().sum().sum()

if nulos_totales > 0:
    print(f'🔴 ADVERTENCIA: {nulos_totales} valores nulos encontrados')
    print(df_clean.isnull().sum())
else:
    print('✅ CERO valores nulos (dataset completo)')

# Completitud por columna
completitud = (1 - df_clean.isnull().sum() / len(df_clean)) * 100
print(f'\nCompletitud por columna:')
for col in df_clean.columns:
    print(f'  {col}: {completitud[col]:.2f}%')

## 3.5 Validación 3: Tipos de datos

In [ ]:
print('VALIDACIÓN 3: TIPOS DE DATOS\n')

tipos_esperados = {
    'timestamp': 'datetime64',
    'avenida': 'object',
    'latitud': 'float',
    'longitud': 'float',
    'velocidad': 'float',
    'densidad': ('int', 'float'),
    'detenciones': ('int', 'float')
}

validaciones_tipo = []

for col, tipo_esperado in tipos_esperados.items():
    if col in df_clean.columns:
        tipo_actual = str(df_clean[col].dtype)
        
        # Verificar tipo
        if isinstance(tipo_esperado, tuple):
            es_valido = any(te in tipo_actual for te in tipo_esperado)
        else:
            es_valido = tipo_esperado in tipo_actual
        
        if es_valido:
            print(f'✅ {col}: {tipo_actual}')
            validaciones_tipo.append(True)
        else:
            print(f'🔴 {col}: Esperado {tipo_esperado}, actual {tipo_actual}')
            validaciones_tipo.append(False)

if all(validaciones_tipo):
    print('\n✅ Todos los tipos de datos son correctos')
else:
    print('\n⚠️  Algunos tipos de datos no coinciden con lo esperado')

## 3.6 Validación 4: Rangos de valores

In [ ]:
print('VALIDACIÓN 4: RANGOS DE VALORES\n')

# Definir rangos válidos
rangos_validos = {
    'velocidad': (0, 120),
    'densidad': (1, 3),
    'detenciones': (0, 30),
    'latitud': (-90, 90),
    'longitud': (-180, 180)
}

validaciones_rango = []

for columna, (minimo, maximo) in rangos_validos.items():
    if columna in df_clean.columns:
        fuera_rango = ((df_clean[columna] < minimo) | (df_clean[columna] > maximo)).sum()
        
        min_actual = df_clean[columna].min()
        max_actual = df_clean[columna].max()
        
        if fuera_rango == 0:
            print(f'✅ {columna}: [{min_actual}, {max_actual}] ✓ dentro de [{minimo}, {maximo}]')
            validaciones_rango.append(True)
        else:
            print(f'🔴 {columna}: {fuera_rango} valores fuera de rango')
            validaciones_rango.append(False)

if all(validaciones_rango):
    print('\n✅ Todos los valores están dentro de rangos válidos')
else:
    print('\n🔴 Se encontraron valores fuera de rangos esperados')

## 3.7 Validación 5: Duplicados

In [ ]:
print('VALIDACIÓN 5: DUPLICADOS\n')

# Duplicados totales
duplicados_totales = df_clean.duplicated().sum()

# Duplicados por evento (timestamp + avenida)
duplicados_evento = df_clean.duplicated(subset=['timestamp', 'avenida']).sum()

print(f'Duplicados totales (filas idénticas): {duplicados_totales}')
print(f'Duplicados por evento (timestamp+avenida): {duplicados_evento}')

if duplicados_totales == 0 and duplicados_evento == 0:
    print('\n✅ CERO duplicados (dataset limpio)')
else:
    print('\n🔴 ADVERTENCIA: Se encontraron duplicados')

## 3.8 Validación 6: Consistencia referencial

In [ ]:
print('VALIDACIÓN 6: CONSISTENCIA REFERENCIAL\n')

# Avenidas permitidas
avenidas_permitidas = ['Av. Chapultepec', 'Av. Lopez Mateos', 'Av. Mexico', 
                       'Anillo Periférico', 'Av. Vallarta', 'Calz. Independencia']

avenidas_actuales = set(df_clean['avenida'].unique())
avenidas_permitidas_set = set(avenidas_permitidas)

avenidas_invalidas = avenidas_actuales - avenidas_permitidas_set
avenidas_faltantes = avenidas_permitidas_set - avenidas_actuales

if len(avenidas_invalidas) == 0:
    print('✅ Todas las avenidas son válidas')
else:
    print(f'🔴 Avenidas inválidas encontradas: {avenidas_invalidas}')

print(f'\nAvenidas en dataset ({len(avenidas_actuales)}):')  
for av in sorted(avenidas_actuales):
    count = (df_clean['avenida'] == av).sum()
    print(f'  - {av}: {count} registros')

if len(avenidas_faltantes) > 0:
    print(f'\n⚠️  Avenidas sin datos: {avenidas_faltantes}')

## 3.9 Validación 7: Temporal

In [ ]:
print('VALIDACIÓN 7: INTEGRIDAD TEMPORAL\n')

if 'timestamp' in df_clean.columns:
    # Verificar que esté ordenado
    ordenado = df_clean['timestamp'].is_monotonic_increasing
    
    if ordenado:
        print('✅ Datos ordenados cronológicamente')
    else:
        print('⚠️  Datos NO están ordenados (se recomienda ordenar)')
    
    # Rango temporal
    fecha_inicio = df_clean['timestamp'].min()
    fecha_fin = df_clean['timestamp'].max()
    duracion_dias = (fecha_fin - fecha_inicio).days
    
    print(f'\nRango temporal:')
    print(f'  Inicio: {fecha_inicio}')
    print(f'  Fin: {fecha_fin}')
    print(f'  Duración: {duracion_dias} días')
    
    # Densidad temporal
    registros_por_dia = len(df_clean) / (duracion_dias + 1)
    print(f'  Registros promedio por día: {registros_por_dia:.2f}')

## 3.10 Validación 8: Estadísticas descriptivas

In [ ]:
print('VALIDACIÓN 8: ESTADÍSTICAS DESCRIPTIVAS\n')
print(df_clean.describe().round(2))

## 3.11 Reporte de calidad final

In [ ]:
print('\n' + '='*80)
print('REPORTE FINAL DE VALIDACIÓN')
print('='*80)

# Resumen de validaciones
validaciones_resumen = {
    '✅ Estructura de datos': len(columnas_faltantes) == 0,
    '✅ Completitud': nulos_totales == 0,
    '✅ Tipos de datos': all(validaciones_tipo),
    '✅ Rangos de valores': all(validaciones_rango),
    '✅ Sin duplicados': duplicados_totales == 0,
    '✅ Consistencia referencial': len(avenidas_invalidas) == 0,
}

validaciones_ok = sum(validaciones_resumen.values())
total_validaciones = len(validaciones_resumen)

print(f'\nResumen de validaciones: {validaciones_ok}/{total_validaciones}')
print()

for validacion, resultado in validaciones_resumen.items():
    estado = '✅ PASS' if resultado else '❌ FAIL'
    print(f'{validacion}: {estado}')

print('\n' + '='*80)

if validaciones_ok == total_validaciones:
    print('🎉 DATASET VALIDADO Y LISTO PARA ANÁLISIS')
else:
    print(f'⚠️  {total_validaciones - validaciones_ok} validaciones fallaron')

print('\n📊 Estadísticas finales:')
print(f'  Total de registros: {len(df_clean)}')
print(f'  Columnas: {len(df_clean.columns)}')
print(f'  Tamaño en memoria: {df_clean.memory_usage(deep=True).sum() / 1024:.2f} KB')

## 3.12 Próximos pasos

In [ ]:
print('\n' + '='*80)
print('PRÓXIMOS PASOS')
print('='*80)
print('''\n✅ Dataset validado y listo para análisis\n
📋 Siguiente fase: Análisis Exploratorio (EDA)

1. Ejecutar: Exploratory_Data_Analysis.ipynb
   - Estadísticas por avenida y horario
   - Visualizaciones de distribuciones
   - Correlaciones entre variables
   - Identificación de patrones

2. Después: Feature Engineering
   - Crear variables predictivas
   - Calcular dinámicas del tráfico
   - Ingeniería de características para TSI

3. Finalmente: Construcción de métrica TSI
   - Fórmula matemática
   - Validación y calibración
   - Pruebas de predicción
''')